# Trascrivi Lezioni — Google Colab

Turn your lectures (audio/video **files** or **links** like YouTube/Webex) into
cleaned, translated **Word documents** — entirely in the cloud. **Nothing to install**
and you don't need to know anything about programming. Powered by Google Gemini.

---

## First time with Colab? Read this (1 minute)
- This page is a **Google Colab notebook**: it runs in your browser, for free, on
  Google's computers. You don't install anything.
- It is made of **cells** (boxes). Some are instructions (like this one), others are
  code. You run a code cell by clicking the **▶ (play)** button on its left, or by
  pressing **Shift+Enter**.
- **Just run the cells in order, from top to bottom.** Click ▶ on the first code cell,
  wait until it finishes (a small **green ✓** appears and the spinner stops), then move
  to the next. You never need to read or edit the code.
- The first time you run a cell, Google may ask you to **authorize / "Run anyway"** —
  that's normal, confirm it.
- Tip: if nothing happens for a while, the cell is still working — just wait.

---

## Step 1 — Get your free Google Gemini API key
The "API key" is a personal password that lets the app use Google Gemini (free).

1. Open **https://aistudio.google.com/apikey** and **sign in** with your Google account.
2. Click **Create API key** (top right).
3. Give the key a **name** (the suggested one, *Gemini API Key*, is fine).
4. Choose a **project**: if you don't have one, click **Create project**, give it a name
   and create it; if you already have one, select it.
5. The key appears on screen — **copy it** (it's a long string like `AIza...`).

Keep it private: it's like a password.

## Step 2 — Give the key to this notebook (do it once)
**Recommended — store it as a Colab Secret (stays hidden):**
1. On the **left side** of this page, click the **🔑 key icon** ("Secrets").
2. Click **＋ Add new secret**.
3. In **Name** type exactly: `GOOGLE_API_KEY`
4. In **Value** paste the key you copied.
5. Turn **ON** the **Notebook access** switch.

That's it — *Step 2* below will pick it up automatically.
*(If you skip this, no problem: when you run Step 2 it will simply ask you to paste the key.)*

## Step 3 — Add your lectures
Two ways (you can mix them):
- **Upload files:** run the *Step 3* cell and click **Choose Files**, then pick your
  lecture videos/audio (`.mp4`, `.m4a`, ...). For **Webex**, also upload a `cookies.txt`.
- **Or drag & drop:** click the **📁 Files** icon on the left and drag your files into it.
- **Links:** you don't upload anything — just paste the YouTube/Webex link in *Step 4*.

You never type file names: the notebook finds your uploaded files by itself.

## Step 4 — Options
A little form: paste any **links** (separated by spaces) and choose the **languages** of the
Word documents (default Italian + English). The subject of each lecture is detected automatically.

## Step 5 — Transcribe
Run it and wait. When it finishes, your Word documents are offered to you as a **`.zip`**
to download. (It can take several minutes per lecture — that's Google doing the work.)

---
**Webex note:** a `cookies.txt` is needed only for Webex "personal room" recordings
(because they require you to be logged in). Export it from your browser with a
"Get cookies.txt" extension, then upload it in Step 3. Public YouTube videos don't need it.

**Privacy:** your audio and transcripts are sent to Google Gemini for processing. Only
transcribe recordings you are allowed to use. A free CPU runtime is enough (no GPU needed).


In [ ]:
#@title Step 1 — Install (run once) { display-mode: "form" }
# Installs the transcriber + yt-dlp in this Colab session. ffmpeg is already on Colab.
!pip install -q "git+https://github.com/GiorgioAlo/lecture-transcriber"
print("Installed. Move on to Step 2.")


In [ ]:
#@title Step 2 — Load your Gemini API key { display-mode: "form" }
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key (hidden): ")
    print("API key set for this session.")


In [ ]:
#@title Step 3 — Add your lectures (upload files) { display-mode: "form" }
# Click "Choose Files" and pick your lecture videos/audio (mp4, m4a, ...).
# For Webex links, also upload a cookies.txt here.
# (Alternative: drag files into the Files panel on the left — they are found too.)
from google.colab import files
uploaded = files.upload()
if uploaded:
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Nothing uploaded here. If you dragged files into /content, that is fine too.")


In [ ]:
#@title Step 4 — Options { display-mode: "form" }
#@markdown Paste one or more **links** (YouTube/Webex), separated by spaces. Leave empty if you only use uploaded files.
LINKS = "" #@param {type:"string"}
#@markdown Output document **languages** (space-separated, e.g. `it en` or `it en fr`):
LANGUAGES = "it en" #@param {type:"string"}
print("Links:", LINKS or "(none)", "| Languages:", LANGUAGES)
#@markdown *(The subject/course of each lecture is detected automatically — nothing to set here.)*


In [ ]:
#@title Step 5 — Transcribe (then download the .zip) { display-mode: "form" }
import os, shutil
from pathlib import Path
from trascrittore import Job, esegui, config
from trascrittore.progress import ReporterConsole

key = os.environ.get("GOOGLE_API_KEY", "")
assert key, "Run Step 2 (API key) first."

OUTPUT_DIR = "/content/transcriptions"
WORK = "/content/.cache"
MEDIA = (".mp4", ".m4a", ".mp3", ".wav", ".aac", ".ogg", ".flac", ".mkv", ".mov", ".webm")

content = Path("/content")
file_lezioni = sorted(p for p in content.iterdir() if p.is_file() and p.suffix.lower() in MEDIA)
links = [u for u in LINKS.replace(",", " ").split() if u.lower().startswith(("http://", "https://"))]
cookies = "/content/cookies.txt" if Path("/content/cookies.txt").exists() else None
langs = LANGUAGES.split() or ["it", "en"]

sources = [("file", str(p)) for p in file_lezioni] + [("link", u) for u in links]
if not sources:
    raise SystemExit("No lectures found. Upload files in Step 3 (or drag them into /content), or add links in Step 4.")

print("Lectures to process:")
for kind, s in sources:
    print("  -", Path(s).name if kind == "file" else s)

out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
saved, failed = [], 0
for i, (kind, src) in enumerate(sources, 1):
    name = Path(src).stem if kind == "file" else "Lecture"
    print()
    print(f"=== [{i}/{len(sources)}] {name} ===")
    job = Job(
        sorgente=Path(src) if kind == "file" else None,
        url=src if kind == "link" else None,
        nome=name, lingue_output=langs, cookies=cookies,
    )
    try:
        ris = esegui(job, key, ReporterConsole(config.FASI), Path(WORK))
    except Exception as e:
        print("  [x] failed:", e); failed += 1; continue
    for d in ris.documenti:
        dest = out / Path(d).name
        shutil.copy2(d, dest); saved.append(dest)
        print("  [doc]", dest)

print()
msg = f"Done. {len(saved)} document(s) saved."
if failed:
    msg += f" ({failed} failed)"
print(msg)

if saved:
    z = shutil.make_archive("/content/transcriptions", "zip", OUTPUT_DIR)
    from google.colab import files as _dl
    _dl.download(z)
